In [ ]:
class Robot:
    def __init__ (self, position, velocity, p_x_max_val, p_y_max_val):
        #print(f"Creation of a new robot on position = {position}, velocity = {velocity}, p_x_max_val = {p_x_max_val}, p_y_max_val = {p_y_max_val}")
        self.second = 0
        self.x = position[0]
        self.y = position[1]
        self.x_velocity = velocity[0]
        self.y_velocity = velocity[1]
        self.x_max = p_x_max_val
        self.y_max = p_y_max_val

    def move(self):
        #print(f"((self.x_max + self.x + self.x_velocity) % (self.x_max - 1) = ({self.x_max} - 1 + {self.x} + {self.x_velocity}) % ({self.x_max}) = ")
        self.x = (self.x_max + self.x + self.x_velocity) % (self.x_max)
        #print(f" = {self.x}")
        #print(f"((self.y_max + self.y + self.y_velocity) % (self.y_max - 1) = ({self.y_max} - 1 + {self.y} + {self.y_velocity}) % ({self.y_max})")
        self.y = (self.y_max + self.y + self.y_velocity) % (self.y_max)
        #print(f"{self.y}")
        self.second += 1

    def get_coordinates(self):
        return (int(self.x), int(self.y))

    def __str__(self):
        return f"Position ({self.x},{self.y}), Velocity ({self.x_velocity},{self.y_velocity}), Second = {self.second}, Field = ({self.x_max},{self.y_max}) "

In [381]:
def calculate_q(p_list, start_x, start_y, end_x, end_y):
    cntr = 0
    for y in range(start_y, end_y):
        for x in range(start_x, end_x):
            if p_list[x][y] != '.':
                cntr += int(p_list[x][y])
    return cntr

In [549]:
class Field:
    def __init__(self, p_x_max_val, p_y_max_val):
        self.x_max = p_x_max_val
        self.y_max = p_y_max_val
        self.l_robots = []
        self.area = []

        for x in range(p_x_max_val):
            self.area.append([])
            for y in range(p_y_max_val):
                self.area[x].append('.') 

    def add_robot(self, p_robot):
        self.l_robots.append(p_robot)

        robot_coordinates = p_robot.get_coordinates()        
        
        if self.area[robot_coordinates[0]][robot_coordinates[1]] == '.':
            self.area[robot_coordinates[0]][robot_coordinates[1]] = '1'
        else:
            self.area[robot_coordinates[0]][robot_coordinates[1]] = str(int(self.area[robot_coordinates[0]][robot_coordinates[1]]) + 1)         

    def one_second_move(self):
        for i in range(len(self.l_robots)):
            (previous_x, previous_y) = self.l_robots[i].get_coordinates()
            self.l_robots[i].move()
            (next_x, next_y) = self.l_robots[i].get_coordinates()
            #print(f"Robot {self.l_robots[i]} makes move from (previous_x, previous_y) = ({previous_x}, {previous_y}) (next_x, next_y) = ({next_x}, {next_y})")

            if self.area[previous_x][previous_y] == '1':
                self.area[previous_x][previous_y] = '.'
            else:
                self.area[previous_x][previous_y] = str(int(self.area[previous_x][previous_y]) - 1)

            if self.area[next_x][next_y] == '.':
                self.area[next_x][next_y] = '1'
            else:
                self.area[next_x][next_y] = str(int(self.area[next_x][next_y]) + 1)    
                
    def calculate_safety_factor(self):
        start_x = 0
        start_y = 0
        end_x = self.x_max // 2
        end_y = self.y_max // 2
        self.q1 = calculate_q(self.area, start_x, start_y, end_x, end_y)
        #print(f"start_x = {start_x}, start_y = {start_y}, end_x = {end_x}, end_y = {end_y}, q1 = {q1}")


        start_x = self.x_max // 2 + 1
        start_y = 0
        end_x = self.x_max
        end_y = self.y_max // 2
        self.q2 = calculate_q(self.area, start_x, start_y, end_x, end_y)
        #print(f"start_x = {start_x}, start_y = {start_y}, end_x = {end_x}, end_y = {end_y}, q2 = {q2}")

        start_x = 0
        start_y = self.y_max // 2 + 1
        end_x = self.x_max // 2
        end_y = self.y_max   
        self.q3 = calculate_q(self.area, start_x, start_y, end_x, end_y)
        #print(f"start_x = {start_x}, start_y = {start_y}, end_x = {end_x}, end_y = {end_y}, q3 = {q3}")

        start_x = self.x_max // 2 + 1
        start_y = self.y_max // 2 + 1
        end_x = self.x_max
        end_y = self.y_max           
        self.q4 = calculate_q(self.area, start_x, start_y, end_x, end_y)
        #print(f"start_x = {start_x}, start_y = {start_y}, end_x = {end_x}, end_y = {end_y}, q4 = {q4}")

        #print(f"q1 = {q1}, q2 = {q2}, q3 = {q3}, q4 = {q4}")
        
        return self.q1 * self.q2 * self.q3 * self.q4

    def longest_series(self):

        max_series = 0
        
        for y in range(self.y_max):
            result = 0    
            for x in range(self.x_max):
                if self.area[x][y] != '.':
                    result += 1
                    if result > max_series:
                        max_series = result

                if self.area[x][y] == '.':
                    result = 0
                    
        return max_series
    
    def __str__(self):
        result = ''
        for y in range(self.y_max):
            for x in range(self.x_max):
                result += self.area[x][y]
            result += '\n'
        return result

In [383]:
def input_row_parse(p_row):
    (t_coordinates, t_velocity) = p_row.split(' ')
    
    t_coordinates = t_coordinates.strip('p=')
    t_coordinates = t_coordinates.split(',')
    t_coordinates = [int(x) for x in t_coordinates]
    
    t_velocity = t_velocity.strip('v=\n')
    t_velocity = t_velocity.split(',')
    t_velocity = [int(x) for x in t_velocity]
    
    return(t_coordinates, t_velocity)

In [384]:
def input_reader(file_name, p_x_max, p_y_max):
    l_input = []
    with open(file_name,'r') as file:
        for row in file:
            (coordinates, velocity) = input_row_parse(row)
            v_robot = Robot(coordinates, velocity, p_x_max, p_y_max)
            #print(v_robot)
            l_input.append(v_robot)
    return l_input

In [516]:
file_name = 'input_sample.txt'
p_x_max_sample = 11
p_y_max_sample = 7
p_x_max = 101
p_y_max = 103

p_robots = input_reader(file_name, p_x_max_sample, p_y_max_sample)

field = Field(p_x_max_sample, p_y_max_sample)
for r in p_robots:
    field.add_robot(r)

for i in range(100):
    field.one_second_move()

print(field)

print(field.calculate_safety_factor())

......2..1.
...........
1..........
.11........
.....1.....
...12......
.1....1....

12


In [252]:
test_robot = Robot((2,4), (2, -3), p_x_max_sample, p_y_max_sample)
field = Field(p_x_max_sample, p_y_max_sample)
field.add_robot(test_robot)
print(field)

...........
...........
...........
...........
..1........
...........
...........



In [518]:
file_name = 'input.txt'

p_x_max = 101
p_y_max = 103

p_robots = input_reader(file_name, p_x_max, p_y_max)

field = Field(p_x_max, p_y_max)
for r in p_robots:
    field.add_robot(r)

for i in range(100):
    field.one_second_move()

#print(field)

print(field.calculate_safety_factor())

225648864


Part 1 Result 225648864

In [556]:
####################################################################################

file_name = 'input.txt'
output_file = 'output.txt'


p_x_max = 101
p_y_max = 103

p_robots = input_reader(file_name, p_x_max, p_y_max)

field = Field(p_x_max, p_y_max)
for r in p_robots:
    field.add_robot(r)



In [ ]:
previous = set()

#i = 0
sf = -1

for i in range(10000):
    previous.add(sf)
    field.one_second_move()
    #print (f"On second {i} the field is:")
    #print(field)
    sf = field.calculate_safety_factor()
    if sf in previous:
        print(f"for i = {i}")
        print(field)
    

#print (i)

In [ ]:
for i in range(234):
    field.one_second_move()
    print(field)

In [ ]:
with open(output_file, 'w') as output:

    for i in range(1, 10000):
        field.one_second_move()
        if field.longest_series() > 10:
            output.write(f'\n--{i}------------------------------------------------------------\n')
            output.write(field.__str__())
            output.write('\n\n')

    output.close()

In [557]:
with open('output_multiple', 'w') as output:

    for i in range(10000):
        if i >= 5000:
            field.one_second_move()
            output.write(f'\n--{i}------------------------------------------------------------\n')
            output.write(field.__str__())
            output.write('\n\n')

    output.close

Result for part 2 is 7847